# API Documentation Agent

This notebook walks through the full pipeline: it downloads and chunks an OpenAPI specification, ingests those chunks into a Vertex AI Search data store, and launches an interactive Gradio chat interface where you can ask natural-language questions about the API and receive answers grounded in the official documentation.

The Kubernetes API is used as the default example, but any OpenAPI 2.0/3.0 spec (URL or local file, JSON or YAML) works.

In [ ]:
!pip install -q uv
!uv pip install -q --system gradio google-cloud-discoveryengine google-cloud-aiplatform==1.71.1 vertexai requests pyyaml 'mcp[cli]'

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
GCP_PROJECT_ID = "your-project-id"                # @param {type:"string"}
GCP_LOCATION = "global"                            # @param {type:"string"}
GEMINI_LOCATION = "us-central1"                   # @param {type:"string"}
VERTEX_SEARCH_DATA_STORE_ID = "your-data-store-id"  # @param {type:"string"}
API_SPEC_URL = "https://raw.githubusercontent.com/kubernetes/kubernetes/v1.36.0/api/openapi-spec/swagger.json"  # @param {type:"string"}
API_NAME = "Kubernetes"                            # @param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
os.environ["GCP_LOCATION"] = GCP_LOCATION
os.environ["GEMINI_LOCATION"] = GEMINI_LOCATION
os.environ["VERTEX_SEARCH_DATA_STORE_ID"] = VERTEX_SEARCH_DATA_STORE_ID
os.environ["API_NAME"] = API_NAME

## Step 1: Ingest API Documentation

This parses the OpenAPI spec and writes one chunk per endpoint operation and one per schema definition to `data/<name>_chunks.jsonl`.

In [ ]:
!python -m src.ingest --spec {API_SPEC_URL} --name {API_NAME.lower()}

## Step 2: Upload to Vertex AI Search

After ingestion, `data/kubernetes_chunks.jsonl` (or `data/<name>_chunks.jsonl`) is ready to import.

1. Open the [Vertex AI Search console](https://console.cloud.google.com/gen-app-builder/data-stores).
2. Create a new data store → **Structured Data** → **JSONL with document IDs**.
3. Import the JSONL file via Cloud Storage or direct upload.
4. Wait for indexing to complete (a few minutes).
5. Copy the **Data Store ID** and paste it into `VERTEX_SEARCH_DATA_STORE_ID` in the config cell above, then re-run that cell.

## Step 3: Launch the Documentation Agent

In [ ]:
import os
os.chdir("/content/api-rag")  # adjust if repo is cloned elsewhere

from src.app import demo
demo.launch(share=True)